In [20]:
from transformers import BertTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments, BertForMaskedLM
from datasets import load_dataset
import datasets
import torch
import re
from nltk.tokenize import sent_tokenize
import nltk
import accelerate
from google_cloud_save import gcs_get_dataset, upload_folder
import os

nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ryanm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
#Load dataset and choose the model we want to use
#TODO Replace gcs_credentials path with your own service account information json file, contact Ryan for help.
gcs_credentaials = "nlp-research-sp26-8499634f1c62.json"

#load the dataset into a hugginface datasets object
dataset = gcs_get_dataset(credentials_path=gcs_credentaials, bucket_name="project3102-data-bucket", data_blob_path="sample_1950s_1960s.jsonl")

model_name = "bert-base-uncased"


Dataset({
    features: ['filename', 'decade', 'text'],
    num_rows: 100
})

In [6]:
dataset[:1]

{'filename': ['mag_1957_186676.txt'],
 'decade': ['1950s'],
 'text': ['Both on Capitol Hill and in the trouble spots around the world , the Administration last week was fighting the battle for U.S. foreign-policy objectives . During the week U.S. diplomats : <P> Urged the Senate , despite mounting opposition by Old Guard Republicans , to ratify the 80-nation treaty to create an International Atomic Energy Agency , key element in Ike \'s 1953 atoms-for-peace proposal before the United Nations . Among the objections : approval would involve the U.S. in a giant " giveaway " of atomic secrets ; Red China might be expected to join and benefit , etc . Secretary of State Dulles , in appearances before two Senate committees , flatly denied the charges and warned the Senators that rejection of the program- " a native American product " that has " caught the imagination of the world " - would be " disastrous " to U.S. prestige and influence . <P> Announced , as the West German Bundestag called o

In [ ]:
# get all dates from dataset, this function currently does years, will need to update to decades
def get_date_tokens(dataset: datasets.Dataset):
    unique_dates = list(set(sorted(dataset['decade'])))
    unique_dates = [str(d).removesuffix('s') for d in unique_dates]
    custom_date_tokens = [f"<decade_{d}>" for d in unique_dates]
    return custom_date_tokens

['1960', '1950']


['<decade_1960>', '<decade_1950>']

In [8]:
#create the tokenizer and add custom tokens
#extra_special_tokens tag is for any non-standard special tokens so we'll use it for all the dates
date_tokens = get_date_tokens(dataset)
tokenizer = BertTokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({'extra_special_tokens' : date_tokens})

['1960', '1950']


2

In [15]:
#append date tokens to the start of all sentences and create simplified dataset

#1. create a dict to hold samples
sentenceData = {'text': []}
#2. iterate entries
for entry in dataset:
    text = entry['text'] # type: ignore
    date = entry['decade'] # type: ignore
    date = str(date).removesuffix('s')
    #3. split entry into sentences.
    for sentence in sent_tokenize(text):
        #4. append date token, limit sentence length to the bert maximum input size, and add to new dataset
        sentence = f'<year_{date}> '+ sentence
        sentence = sentence[:min(512, len(sentence))]
        sentenceData['text'].append(sentence)

#5. create cleaned dataset from sample dict
timestamped_dataset = datasets.Dataset.from_dict(sentenceData)


In [16]:
#tokenizer function used to map cleaned samples to tokenizer token ids
def tokenize_data(examples):
    result = tokenizer(examples["text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result



In [17]:
# Create Data collator for Masked Language Modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

# split dataset into train and validation sets
split = timestamped_dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split['train']
val_dataset   = split['test']

print(f"Train samples: {len(train_dataset)}")
print(f"Val   samples: {len(val_dataset)}")

# tokenize datasets
train_dataset = train_dataset.map(tokenize_data, batch_size=64, batched=True)
val_dataset   = val_dataset.map(tokenize_data,   batch_size=64, batched=True)

Train samples: 8115
Val   samples: 902


Map: 100%|██████████| 902/902 [00:00<00:00, 12554.92 examples/s]


In [18]:
# Load pre-trained model and resize to the custom tokenizer
model = BertForMaskedLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 888.13it/s, Materializing param=cls.predictions.transform.dense.weight]                 
BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(30524, 768, padding_idx=0)

In [ ]:
#Train using 
training_args = TrainingArguments(
    output_dir="./NYT_pretrained_model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    #saving model based on eval_loss
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=100,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

# Pretrain the model
trainer.train()

#give this training run a name
model_name = "100-Samples-Training"

#save model
save_path = f"./NYT_pretrained_model/{model_name}-best"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
c:\Users\ryanm\Comp Sci Work\Semantics-Research\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.871656,1.711431
2,1.643940,1.718669
3,1.628569,1.773510


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.58it/s]
c:\Users\ryanm\Comp Sci Work\Semantics-Research\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.61it/s]
c:\Users\ryanm\Comp Sci Work\Semantics-Research\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight

('./NYT_pretrained_model/best\\tokenizer_config.json',
 './NYT_pretrained_model/best\\tokenizer.json')

In [34]:
#upload model to GCS
model_name = "100-Samples-Training"
local_dir = "100-Samples-Training-best"
bucket_name = "project3102-model-bucket"
destination_blob_prefix = "Training-Tests/" # Folder path in GCS
print(save_path)

upload_folder(credentials_path=gcs_credentaials, bucket_name=bucket_name, destination_blob_prefix=destination_blob_prefix, local_dir=local_dir)

./Semantics-Research/100-Samples-Training-best
Uploaded 100-Samples-Training-best\config.json to gs://project3102-model-bucket/Training-Tests/config.json
Uploaded 100-Samples-Training-best\model.safetensors to gs://project3102-model-bucket/Training-Tests/model.safetensors
Uploaded 100-Samples-Training-best\tokenizer.json to gs://project3102-model-bucket/Training-Tests/tokenizer.json
Uploaded 100-Samples-Training-best\tokenizer_config.json to gs://project3102-model-bucket/Training-Tests/tokenizer_config.json
Uploaded 100-Samples-Training-best\training_args.bin to gs://project3102-model-bucket/Training-Tests/training_args.bin
All model files uploaded to Google Cloud Storage.
